In [52]:
import pandas as pd

# Load df_combined.csv into a DataFrame called df
df = pd.read_csv("/Users/katinkakurz/projects/Thesis/data/raw/df_combined.csv")

In [53]:
df.head()

,Date,Title,Text,source,row_id
0,2025-08-01,Bereitet der Westen die Entmachtung oder sogar...,Der Streit um das Nationale Anti-Korruptionsbü...,Antispiegel,1
1,2025-08-01,EU-Kommission hat Textnachrichten zum Kauf der...,Die New York Times versucht seit langem vor Ge...,Antispiegel,2
2,2025-08-01,Wahlkommission rechtfertigt Einmischung der EU...,Ende September stehen in Moldawien Parlamentsw...,Antispiegel,3
3,2025-08-02,Fordert Russland wirklich die Vernichtung alle...,Ich bin auf die Geschichte zuerst bei RT-DE ge...,Antispiegel,4
4,2025-08-02,Der Spiegel macht mal wieder Berichterstattung...,"Wer auch nicht-westliche Medien liest, der ist...",Antispiegel,5


In [54]:
import numpy as np

# Ensure there is a 'data_source' column; adjust if named differently
source_col = "source"

# Filter sources with at least 5 articles
source_counts = df[source_col].value_counts()
eligible_sources = source_counts[source_counts >= 5].index

# Subset dataframe to only eligible sources
eligible_df = df[df[source_col].isin(eligible_sources)].copy()

# Sample at least 5 articles per source first
frames = []
for source in eligible_sources:
    source_df = eligible_df[eligible_df[source_col] == source]
    sample_size = min(5, len(source_df))
    sampled = source_df.sample(n=sample_size, random_state=42)
    frames.append(sampled)

base_sample = pd.concat(frames, ignore_index=True)
remaining_needed = max(0, 50 - len(base_sample))

# For the rest, sample randomly from eligible_df excluding already selected rows
if remaining_needed > 0:
    remaining_df = eligible_df.drop(base_sample.index)
    additional_sample = remaining_df.sample(
        n=min(remaining_needed, len(remaining_df)), random_state=42, replace=False
    )
    trial_df = pd.concat([base_sample, additional_sample], ignore_index=True)
else:
    trial_df = base_sample

# If trial_df is still larger than 100 (due to many eligible sources), subsample to 100
if len(trial_df) > 100:
    trial_df = trial_df.sample(n=100, random_state=42, replace=False).reset_index(drop=True)
else:
    trial_df = trial_df.reset_index(drop=True)

print(f"trial_df shape: {trial_df.shape}")

trial_df shape: (50, 5)


In [55]:
trial_df

,Date,Title,Text,source,row_id
0,2026-01-20,Huawei und ZTE sollen raus aus dem Netz,Die EU-Kommission will die europäische Infrast...,Tagesschau,18204
1,2025-12-09,Mindestens 30 Verletzte durch Erdbeben in Japan,Regierungschefin Takaichi ruft zur Vorsicht au...,Tagesschau,17965
2,2025-09-13,Oppositionspolitiker in der Türkei festgenommen,Die Istanbuler Staatsanwaltschaft hat 48 Bezir...,Tagesschau,14833
3,2025-12-12,Will Trump das Erdöl?,Die Beschlagnahmung des Tankers vor Venezuela ...,Tagesschau,17770
4,2025-10-15,Junge Leser retten die Buchbranche,Das Interesse an Büchern sinkt bei den Älteren...,Tagesschau,16994
5,2025-12-29,IAEA: Technischer Waffenstillstand am AKW Sapo...,Die Internationale Atomenergiebehörde (IAEA) h...,RT_de,5792
6,2025-10-14,Politischer Druck: Niederlande verstaatlichen ...,Die niederländische Regierung hat die Kontroll...,RT_de,7967
7,2026-01-18,Plötzliche Kehrtwende: Bundeswehr zieht sich l...,Alle 15 Soldaten der Bundeswehr flogen heute u...,RT_de,5344
8,2025-09-15,"""Fasziniert"" vom Mord: Tages-Anzeiger vergleic...","Tages-Anzeger: ""Sniper in der Popkultur: Wesha...",RT_de,8798
9,2026-01-14,Politische Kultur der USA treibt Annexion Grön...,Von Timofei Bordatschow Die Vereinigten Staate...,RT_de,5456


In [56]:
trial_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Date    50 non-null     str  
 1   Title   50 non-null     str  
 2   Text    50 non-null     str  
 3   source  50 non-null     str  
 4   row_id  50 non-null     int64
dtypes: int64(1), str(4)
memory usage: 2.1 KB


In [57]:
from pathlib import Path

prompt_candidates = [Path("promptv1.txt"), Path("1b_TopicClassification/promptv1.txt")]
prompt_path = next((path for path in prompt_candidates if path.exists()), None)
if prompt_path is None:
    raise FileNotFoundError("promptv1.txt not found in the current directory or in 1b_TopicClassification/")

prompt = prompt_path.read_text(encoding="utf-8").strip()
print(f"Loaded prompt from: {prompt_path}")
print(prompt)

Loaded prompt from: promptv1.txt
Prompt: Du bist ein Experte für die Klassifikation deutscher politischer Nachrichten. Deine Aufgabe besteht darin, den Inhalt des dir vorliegenden Nachrichtenartikels sorgfältig zu untersuchen. Deine primäre Aufgabe ist es, den Artikel auf Grundlage seiner vorherrschenden Themen und seines Gegenstands in die folgenden spezifischen Kategorien einzuordnen:
UKRAINE-KONFLIKT & KRIEG (Krieg in der Ukraine, Putin, Selenskyj, Russland, NATO, Militärhilfe, Friedensverhandlungen, Waffenlieferungen, Sanktionen im Zusammenhang mit dem Ukraine-Konflikt)
RUSSLAND, GAS & SANKTIONEN (Russische Gasversorgung, Nord Stream, Energiesanktionen, russische Wirtschaft, Handelsbeziehungen mit Russland, Energieabhängigkeiten)
NAHER OSTEN (ISRAEL, GAZA, HAMAS) (Israel-Palästina-Konflikt, Gaza, Hamas, Netanyahu, Geiseln, humanitäre Lage, Iran-Spannungen, Politik im Nahen Osten)
MIGRATION & ASYL (Asylsuchende, Flüchtlinge, Migrationspolitik, Abschiebungen, Grenzsicherung, Integrat

## Prompt input creation

In [58]:
import json
import pandas as pd

# Work on a copy; do not modify the original df
work = trial_df.copy()
title_col = "Title" if "Title" in work.columns else "title"

# Ensure we have string columns; fill NaN with "" and strip whitespace
for col in [title_col, "Text"]:
    if col in work.columns:
        work[col] = work[col].fillna("").astype(str).str.strip()
    else:
        work[col] = ""

n_original = len(work)



# Build the text sent to the model
work["input_text"] = (
    "Title: " + work[title_col] + "\n\nText: " + work["Text"]
)

# Build the full JSON object per row for the batch API
batch_lines = []
for _, row in work.iterrows():
    obj = {
        "custom_id": row["row_id"],
        "method": "POST",
        "url": "/v1/responses",
        "body": {
            "model": "gpt-5-mini",
            "instructions": prompt,
            "input": row["input_text"],
        },
    }
    batch_lines.append(json.dumps(obj, ensure_ascii=False))

# Write JSONL (UTF-8)
with open("batch_input.jsonl", "w", encoding="utf-8") as f:
    for line in batch_lines:
        f.write(line + "\n")

In [59]:
# Validate the JSON objects and preview the first rows
required_top_keys = {"custom_id", "method", "url", "body"}
required_body_keys = {"model", "instructions", "input"}

parsed_rows = []
invalid_rows = []

for idx, line in enumerate(batch_lines, start=1):
    try:
        obj = json.loads(line)
        missing_top = sorted(required_top_keys - set(obj.keys()))
        body = obj.get("body", {})
        body_keys = set(body.keys()) if isinstance(body, dict) else set()
        missing_body = sorted(required_body_keys - body_keys)

        if missing_top or missing_body:
            invalid_rows.append({
                "row_number": idx,
                "custom_id": obj.get("custom_id"),
                "missing_top_keys": ", ".join(missing_top),
                "missing_body_keys": ", ".join(missing_body),
            })
        else:
            parsed_rows.append(obj)
    except json.JSONDecodeError as e:
        invalid_rows.append({
            "row_number": idx,
            "custom_id": None,
            "missing_top_keys": "invalid JSON",
            "missing_body_keys": str(e),
        })

with open("batch_input.jsonl", "r", encoding="utf-8") as f:
    jsonl_line_count = sum(1 for _ in f)

creation_worked = (
    len(batch_lines) == n_original
    and len(parsed_rows) == len(batch_lines)
    and len(invalid_rows) == 0
    and jsonl_line_count == len(batch_lines)
)

print(f"Original rows:          {n_original}")
print(f"JSON strings created:   {len(batch_lines)}")
print(f"Lines in JSONL file:    {jsonl_line_count}")
print(f"Valid JSON objects:     {len(parsed_rows)}")
print(f"Invalid JSON objects:   {len(invalid_rows)}")
print(f"Creation worked:        {creation_worked}")

if parsed_rows:
    preview_df = pd.DataFrame([
        {
            "custom_id": obj["custom_id"],
            "method": obj["method"],
            "url": obj["url"],
            "model": obj["body"]["model"],
            "input_preview": obj["body"]["input"][:120].replace("\n", " ") + ("..." if len(obj["body"]["input"]) > 120 else ""),
        }
        for obj in parsed_rows[:5]
    ])

    print("\nFirst rows preview:")
    display(preview_df)

    print("\nFirst JSON object:")
    print(json.dumps(parsed_rows[0], ensure_ascii=False, indent=2))

if invalid_rows:
    print("\nRows with JSON issues:")
    display(pd.DataFrame(invalid_rows))

Original rows:          50
JSON strings created:   50
Lines in JSONL file:    50
Valid JSON objects:     50
Invalid JSON objects:   0
Creation worked:        True

First rows preview:


,custom_id,method,url,model,input_preview
0,18204,POST,/v1/responses,gpt-5-mini,Title: Huawei und ZTE sollen raus aus dem Netz...
1,17965,POST,/v1/responses,gpt-5-mini,Title: Mindestens 30 Verletzte durch Erdbeben ...
2,14833,POST,/v1/responses,gpt-5-mini,Title: Oppositionspolitiker in der Türkei fest...
3,17770,POST,/v1/responses,gpt-5-mini,Title: Will Trump das Erdöl? Text: Die Beschl...
4,16994,POST,/v1/responses,gpt-5-mini,Title: Junge Leser retten die Buchbranche Tex...



First JSON object:
{
  "custom_id": 18204,
  "method": "POST",
  "url": "/v1/responses",
  "body": {
    "model": "gpt-5-mini",
    "instructions": "Prompt: Du bist ein Experte für die Klassifikation deutscher politischer Nachrichten. Deine Aufgabe besteht darin, den Inhalt des dir vorliegenden Nachrichtenartikels sorgfältig zu untersuchen. Deine primäre Aufgabe ist es, den Artikel auf Grundlage seiner vorherrschenden Themen und seines Gegenstands in die folgenden spezifischen Kategorien einzuordnen:\nUKRAINE-KONFLIKT & KRIEG (Krieg in der Ukraine, Putin, Selenskyj, Russland, NATO, Militärhilfe, Friedensverhandlungen, Waffenlieferungen, Sanktionen im Zusammenhang mit dem Ukraine-Konflikt)\nRUSSLAND, GAS & SANKTIONEN (Russische Gasversorgung, Nord Stream, Energiesanktionen, russische Wirtschaft, Handelsbeziehungen mit Russland, Energieabhängigkeiten)\nNAHER OSTEN (ISRAEL, GAZA, HAMAS) (Israel-Palästina-Konflikt, Gaza, Hamas, Netanyahu, Geiseln, humanitäre Lage, Iran-Spannungen, Politik

## Execute `gpt-5-mini` on the 50 selected rows

In [60]:
import os
import time
from pathlib import Path

import requests

MODEL_NAME = "gpt-5-mini"
API_URL = "https://api.openai.com/v1/responses"
RESULTS_PATH = "classification_trial_results_gpt5mini.csv"
ERRORS_PATH = "classification_trial_errors_gpt5mini.csv"


def read_env_value(name: str):
    env_value = os.getenv(name)
    if env_value:
        return env_value, "environment variable"

    env_candidates = [Path.cwd() / ".env", Path.cwd().parent / ".env"]
    for env_path in env_candidates:
        if not env_path.exists():
            continue

        for raw_line in env_path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue

            key, value = line.split("=", 1)
            if key.strip() == name:
                return value.strip().strip('"').strip("'"), str(env_path)

    return None, None


def extract_output_text(response_json: dict) -> str:
    output_text = response_json.get("output_text")
    if isinstance(output_text, str) and output_text.strip():
        return output_text.strip()

    text_chunks = []
    for item in response_json.get("output", []):
        for content in item.get("content", []):
            if content.get("type") == "output_text" and isinstance(content.get("text"), str):
                text_chunks.append(content["text"])

    return "\n".join(text_chunks).strip()


api_key, api_key_source = read_env_value("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY not found. Add it to .env or export it in your shell.")

print(f"Loaded API key from: {api_key_source}")

run_df = trial_df.iloc[:50].copy()
title_col = "Title" if "Title" in run_df.columns else "title"
required_cols = ["row_id", title_col, "Text"]
missing_cols = [col for col in required_cols if col not in run_df.columns]
if missing_cols:
    raise KeyError(f"Missing required columns: {missing_cols}")

run_df[title_col] = run_df[title_col].fillna("").astype(str).str.strip()
run_df["Text"] = run_df["Text"].fillna("").astype(str).str.strip()
run_df["input_text"] = "Title: " + run_df[title_col] + "\n\nText: " + run_df["Text"]

session = requests.Session()
session.headers.update(
    {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
)

results = []
errors = []
result_columns = ["row_id", "Date", "source", "title", "Text", "model", "response_id", "classification_text", "raw_response_json"]
error_columns = ["row_id", "title", "Text", "status_code", "error"]

for idx, row in run_df.iterrows():
    payload = {
        "model": MODEL_NAME,
        "instructions": prompt,
        "input": row["input_text"],
    }

    try:
        response = session.post(API_URL, json=payload, timeout=180)
        response.raise_for_status()
        response_json = response.json()

        results.append(
            {
                "row_id": row["row_id"],
                "Date": row.get("Date", ""),
                "source": row.get("source", ""),
                "title": row[title_col],
                "Text": row["Text"],
                "model": MODEL_NAME,
                "response_id": response_json.get("id", ""),
                "classification_text": extract_output_text(response_json),
                "raw_response_json": json.dumps(response_json, ensure_ascii=False),
            }
        )
    except requests.HTTPError as exc:
        try:
            error_body = exc.response.json()
        except ValueError:
            error_body = {"message": exc.response.text[:2000] if exc.response is not None else str(exc)}

        errors.append(
            {
                "row_id": row["row_id"],
                "title": row[title_col],
                "Text": row["Text"],
                "status_code": exc.response.status_code if exc.response is not None else None,
                "error": json.dumps(error_body, ensure_ascii=False),
            }
        )
    except requests.RequestException as exc:
        errors.append(
            {
                "row_id": row["row_id"],
                "title": row[title_col],
                "Text": row["Text"],
                "status_code": None,
                "error": str(exc),
            }
        )

    if (idx + 1) % 5 == 0 or (idx + 1) == len(run_df):
        print(f"Processed {idx + 1}/{len(run_df)} rows")

    time.sleep(0.2)

results_df = pd.DataFrame(results, columns=result_columns)
errors_df = pd.DataFrame(errors, columns=error_columns)

results_df.to_csv(RESULTS_PATH, index=False, encoding="utf-8")
errors_df.to_csv(ERRORS_PATH, index=False, encoding="utf-8")

if not results_df.empty:
    print(f"\nSaved successful classifications to {RESULTS_PATH}")
    display(results_df[["row_id", "source", "title", "Text", "classification_text"]].head())
else:
    print(f"\nNo successful responses were returned. Wrote an empty file to {RESULTS_PATH}")

if not errors_df.empty:
    print(f"\nSaved request errors to {ERRORS_PATH}")
    display(errors_df)
else:
    print(f"\nAll rows completed without HTTP/request errors. Wrote an empty file to {ERRORS_PATH}")


Loaded API key from: /Users/katinkakurz/projects/Thesis/.env
Processed 5/50 rows
Processed 10/50 rows
Processed 15/50 rows
Processed 20/50 rows
Processed 25/50 rows
Processed 30/50 rows
Processed 35/50 rows
Processed 40/50 rows
Processed 45/50 rows
Processed 50/50 rows

Saved successful classifications to classification_trial_results_gpt5mini.csv


,row_id,source,title,Text,classification_text
0,18204,Tagesschau,Huawei und ZTE sollen raus aus dem Netz,Die EU-Kommission will die europäische Infrast...,Thema 1: KRIMINALITÄT & INNERE SICHERHEIT\nEvi...
1,17965,Tagesschau,Mindestens 30 Verletzte durch Erdbeben in Japan,Regierungschefin Takaichi ruft zur Vorsicht au...,Thema 1: SONSTIGES\nEvidenz 1: “mindestens 30 ...
2,14833,Tagesschau,Oppositionspolitiker in der Türkei festgenommen,Die Istanbuler Staatsanwaltschaft hat 48 Bezir...,Thema 1: PARTEIEN & OPPOSITION (AFD-FOKUS)\nEv...
3,17770,Tagesschau,Will Trump das Erdöl?,Die Beschlagnahmung des Tankers vor Venezuela ...,Thema 1: ENERGIE & KLIMAPOLITIK\nEvidenz 1: “r...
4,16994,Tagesschau,Junge Leser retten die Buchbranche,Das Interesse an Büchern sinkt bei den Älteren...,Thema 1: WIRTSCHAFT & INDUSTRIE\nEvidenz 1: “T...



All rows completed without HTTP/request errors. Wrote an empty file to classification_trial_errors_gpt5mini.csv
